## 1️⃣ Instalação de Dependências

In [ ]:
!pip install -q easyocr
!pip install -q opencv-python
!pip install -q opencv-contrib-python
!pip install -q pytesseract

# Tesseract OCR com suporte português
!apt-get update -qq
!apt-get install -qq tesseract-ocr
!apt-get install -qq tesseract-ocr-por
!apt-get install -qq libtesseract-dev


## 2️⃣ Importações e Configuração

In [ ]:
# Importações necessárias
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from pathlib import Path
import easyocr
import pytesseract
from google.colab import drive
from google.colab.patches import cv2_imshow
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (20, 12)
plt.rcParams['font.size'] = 10


## 3️⃣ Conectar Google Drive

In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Configurar pasta de imagens
pasta_imagens = '/content/drive/MyDrive/placas_veiculos'

if not os.path.exists(pasta_imagens):
    print(f"  Pasta não encontrada: {pasta_imagens}")
    print(" Criando pasta...")
    os.makedirs(pasta_imagens, exist_ok=True)
    print(f" Pasta criada: {pasta_imagens}")
    print("\n IMPORTANTE: Faça upload de imagens de carros/motos nesta pasta!")
else:
    print(f"Pasta encontrada: {pasta_imagens}")

# Verificar imagens existentes
extensoes = ['.jpg', '.jpeg', '.png', '.bmp']
imagens_existentes = []

for ext in extensoes:
    imagens_existentes.extend([
        f for f in os.listdir(pasta_imagens)
        if f.lower().endswith(ext.lower())
    ])


## 4️⃣ Sistema Melhorado de Reconhecimento de Placas

In [ ]:
class SistemaReconhecimentoPlacasMelhorado:

    # Type hints for methods added dynamically
    def detectar_placas_melhorado(self, imagem: np.ndarray) -> list: ...
    def _detectar_por_contornos(self, imagem_binaria: np.ndarray, imagem_original: np.ndarray) -> list: ...
    def _detectar_por_bordas(self, imagem_bordas: np.ndarray, imagem_original: np.ndarray) -> list: ...
    def _detectar_por_componentes(self, imagem_binaria: np.ndarray, imagem_original: np.ndarray) -> list: ...
    def _tem_caracteristicas_texto(self, roi: np.ndarray) -> bool: ...
    def _calcular_score_placa(self, roi: np.ndarray, area: float, aspect_ratio: float) -> float: ...
    def _filtrar_placas_candidatas(self, candidatos: list) -> list: ...
    def _validar_com_ocr_preliminar(self, candidatos: list, imagem_original: np.ndarray) -> list: ...
    def _ocr_rapido_easyocr(self, imagem: np.ndarray) -> str: ...
    def _ocr_rapido_tesseract(self, imagem: np.ndarray) -> str: ...
    def _validar_texto_placa(self, texto1: str, texto2: str) -> float: ...
    def processar_imagem(self, caminho_imagem: str) -> dict: ...
    def _ocr_tesseract_completo(self, imagem: np.ndarray) -> str: ...
    def _ocr_easyocr_completo(self, imagem: np.ndarray) -> str: ...
    def _pos_processar_texto(self, texto: str) -> str: ...


    def __init__(self):

        # Configurar EasyOCR
        try:
            self.easyocr_reader = easyocr.Reader(['pt', 'en'], gpu=True)

        except:
            try:
                self.easyocr_reader = easyocr.Reader(['pt', 'en'], gpu=False)
                print("EasyOCR configurado (CPU)")
            except:
                self.easyocr_reader = None
                print("EasyOCR não disponível")

        # Configurações OTIMIZADAS para placas brasileiras
        self.config = {
            # Pré-processamento específico para placas
            'gaussian_kernel': (7, 7),          # Preservar detalhes
            'bilateral_d': 7,                   # Menos suavização
            'bilateral_sigma_color': 80,
            'bilateral_sigma_space': 80,

            # Detecção de bordas mais sensível
            'canny_low': 30,                    # Mais sensível
            'canny_high': 120,                  # Menos restritivo

            # Morfologia para conectar caracteres
            'morph_kernel_size': (2, 2),        # Kernel pequeno
            'morph_rect_kernel': (3, 1),        # Conectar horizontalmente

            # FILTROS ESPECÍFICOS PARA PLACAS BRASILEIRAS
            'placa_aspect_ratio_min': 2.5,      # Placas são largas
            'placa_aspect_ratio_max': 5.5,      # Não muito largas
            'placa_area_min': 800,             # Área mínima
            'placa_area_max': 800000,            # MÁXIMO - evita carros
            'placa_width_min': 80,              # Largura mínima
            'placa_height_min': 20,             # Altura mínima
            'placa_width_max': 400,             # MÁXIMO - evita carros
            'placa_height_max': 120,            # MÁXIMO - evita carros

            # OCR otimizado
            'tesseract_config': '--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',

            # Validação de texto
            'densidade_texto_min': 0.15,         # Mínimo de pixels de texto
            'densidade_texto_max': 0.85,         # Máximo de pixels de texto
            'picos_minimos': 3,                 # Mínimo de caracteres
        }

        print("\n CONFIGURAÇÕES:")
        print(f"    Área de placa: {self.config['placa_area_min']} - {self.config['placa_area_max']} pixels")
        print(f"    Dimensões: {self.config['placa_width_min']}-{self.config['placa_width_max']} x {self.config['placa_height_min']}-{self.config['placa_height_max']}")
        print(f"    Aspect ratio: {self.config['placa_aspect_ratio_min']:.1f} - {self.config['placa_aspect_ratio_max']:.1f}")

    def filtrar_regiao_interesse(self, imagem):
        """Filtrar região onde placas tipicamente aparecem (terço inferior)"""
        h, w = imagem.shape[:2]

        # Criar máscara para região de interesse
        mask = np.zeros((h, w), dtype=np.uint8)

        # Região: 40% a 90% da altura (onde ficam as placas)
        roi_top = int(h * 0.4)
        roi_bottom = int(h * 0.9)

        mask[roi_top:roi_bottom, :] = 255

        # Aplicar máscara
        if len(imagem.shape) == 3:
            imagem_roi = cv2.bitwise_and(imagem, imagem, mask=mask)
        else:
            imagem_roi = cv2.bitwise_and(imagem, mask)

        return imagem_roi, mask

    def preprocessar_para_placas(self, imagem):
        """Pré-processamento otimizado para destacar placas"""
        resultados = {'original': imagem.copy()}

        # 1. SUAVIZAÇÃO MÍNIMA (preservar detalhes)
        suavizada = cv2.bilateralFilter(
            imagem,
            self.config['bilateral_d'],
            self.config['bilateral_sigma_color'],
            self.config['bilateral_sigma_space']
        )
        resultados['suavizada'] = suavizada

        # Converter para escala de cinza
        gray = cv2.cvtColor(suavizada, cv2.COLOR_BGR2GRAY)

        # 2. Múltiplas BINARIZAÇÕES
        # Otsu
        _, bin_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        resultados['bin_otsu'] = bin_otsu

        # Adaptativa
        bin_adaptiva = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
        )
        resultados['bin_adaptiva'] = bin_adaptiva

        # Adaptativa INVERSA (para placas escuras)
        bin_adaptiva_inv = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
        )
        resultados['bin_adaptiva_inv'] = bin_adaptiva_inv

        # 3. DETECÇÃO DE BORDAS OTIMIZADA
        bordas_canny = cv2.Canny(gray, self.config['canny_low'], self.config['canny_high'])
        resultados['bordas_canny'] = bordas_canny

        # 4. OPERAÇÕES MORFOLÓGICAS PARA TEXTO
        kernel_small = cv2.getStructuringElement(cv2.MORPH_RECT, self.config['morph_kernel_size'])
        kernel_horizontal = cv2.getStructuringElement(cv2.MORPH_RECT, self.config['morph_rect_kernel'])

        # Conectar caracteres horizontalmente
        morph_close_horizontal = cv2.morphologyEx(bin_adaptiva, cv2.MORPH_CLOSE, kernel_horizontal)
        resultados['morph_close_horizontal'] = morph_close_horizontal

        # Opening para remover ruído
        morph_opening = cv2.morphologyEx(morph_close_horizontal, cv2.MORPH_OPEN, kernel_small)
        resultados['morph_opening'] = morph_opening

        return resultados


In [ ]:
# Continuação da classe - PARTE 2/3: Métodos de Detecção

def detectar_placas_melhorado(self, imagem):
    """Detecção melhorada com múltiplas estratégias"""
    # ETAPA 1: Filtrar região de interesse
    imagem_roi, mask_roi = self.filtrar_regiao_interesse(imagem)

    # ETAPA 2: Pré-processar
    prep_results = self.preprocessar_para_placas(imagem_roi)

    # ETAPA 3: Múltiplas estratégias de detecção
    candidatos = []

    # Estratégia 1: Contornos
    candidatos.extend(self._detectar_por_contornos(prep_results['morph_opening'], imagem))

    # Estratégia 2: Bordas
    candidatos.extend(self._detectar_por_bordas(prep_results['bordas_canny'], imagem))

    # Estratégia 3: Componentes conectados
    candidatos.extend(self._detectar_por_componentes(prep_results['bin_adaptiva'], imagem))

    # ETAPA 4: Filtrar duplicatas e ranquear
    candidatos_filtrados = self._filtrar_placas_candidatas(candidatos)

    # ETAPA 5: Validação com OCR preliminar
    placas_validadas = self._validar_com_ocr_preliminar(candidatos_filtrados, imagem)

    return placas_validadas[:3]  # Máximo 3 melhores placas

def _detectar_por_contornos(self, imagem_binaria, imagem_original):
    """Detecção por análise de contornos"""
    candidatos = []

    contornos, _ = cv2.findContours(imagem_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contorno in contornos:
        area = cv2.contourArea(contorno)

        # Filtro de área RIGOROSO
        if not (self.config['placa_area_min'] <= area <= self.config['placa_area_max']):
            continue

        x, y, w, h = cv2.boundingRect(contorno)

        # Filtros dimensionais RIGOROSOS
        if not (self.config['placa_width_min'] <= w <= self.config['placa_width_max']):
            continue
        if not (self.config['placa_height_min'] <= h <= self.config['placa_height_max']):
            continue

        # Aspect ratio RIGOROSO
        aspect_ratio = w / float(h)
        if not (self.config['placa_aspect_ratio_min'] <= aspect_ratio <= self.config['placa_aspect_ratio_max']):
            continue

        # Verificar características de TEXTO
        roi = imagem_binaria[y:y+h, x:x+w]
        if self._tem_caracteristicas_texto(roi):
            candidatos.append({
                'bbox': (x, y, x+w, y+h),
                'area': area,
                'aspect_ratio': aspect_ratio,
                'score': self._calcular_score_placa(roi, area, aspect_ratio),
                'metodo': 'Contornos-V2'
            })

    return candidatos

def _detectar_por_bordas(self, imagem_bordas, imagem_original):
    """Detecção usando bordas"""
    candidatos = []

    # Dilatar bordas para formar regiões
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    bordas_dilatadas = cv2.dilate(imagem_bordas, kernel, iterations=2)

    contornos, _ = cv2.findContours(bordas_dilatadas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contorno in contornos:
        area = cv2.contourArea(contorno)

        if not (self.config['placa_area_min'] <= area <= self.config['placa_area_max']):
            continue

        x, y, w, h = cv2.boundingRect(contorno)
        aspect_ratio = w / float(h)

        if (self.config['placa_width_min'] <= w <= self.config['placa_width_max'] and
            self.config['placa_height_min'] <= h <= self.config['placa_height_max'] and
            self.config['placa_aspect_ratio_min'] <= aspect_ratio <= self.config['placa_aspect_ratio_max']):

            candidatos.append({
                'bbox': (x, y, x+w, y+h),
                'area': area,
                'aspect_ratio': aspect_ratio,
                'score': self._calcular_score_placa(imagem_bordas[y:y+h, x:x+w], area, aspect_ratio),
                'metodo': 'Bordas-V2'
            })

    return candidatos

def _detectar_por_componentes(self, imagem_binaria, imagem_original):
    """Detecção por componentes conectados"""
    candidatos = []

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagem_binaria, connectivity=8)

    for i in range(1, num_labels):  # Ignorar background
        x, y, w, h, area = stats[i]

        if not (self.config['placa_area_min'] <= area <= self.config['placa_area_max']):
            continue

        aspect_ratio = w / float(h)

        if (self.config['placa_width_min'] <= w <= self.config['placa_width_max'] and
            self.config['placa_height_min'] <= h <= self.config['placa_height_max'] and
            self.config['placa_aspect_ratio_min'] <= aspect_ratio <= self.config['placa_aspect_ratio_max']):

            component_mask = (labels == i).astype(np.uint8) * 255
            roi = component_mask[y:y+h, x:x+w]

            candidatos.append({
                'bbox': (x, y, x+w, y+h),
                'area': area,
                'aspect_ratio': aspect_ratio,
                'score': self._calcular_score_placa(roi, area, aspect_ratio),
                'metodo': 'Componentes-V2'
            })

    return candidatos

# Adicionar métodos à classe
SistemaReconhecimentoPlacasMelhorado.detectar_placas_melhorado = detectar_placas_melhorado
SistemaReconhecimentoPlacasMelhorado._detectar_por_contornos = _detectar_por_contornos
SistemaReconhecimentoPlacasMelhorado._detectar_por_bordas = _detectar_por_bordas
SistemaReconhecimentoPlacasMelhorado._detectar_por_componentes = _detectar_por_componentes

In [ ]:
# Continuação da classe - PARTE 3/3: Validação e OCR

def _tem_caracteristicas_texto(self, roi):
    """Verificar se região tem características de texto em placas"""
    if roi.size == 0:
        return False

    # Densidade de pixels brancos (texto)
    densidade = np.sum(roi == 255) / roi.size

    if not (self.config['densidade_texto_min'] <= densidade <= self.config['densidade_texto_max']):
        return False

    # Distribuição horizontal (caracteres se distribuem horizontalmente)
    if roi.shape[1] > 10:  # Largura mínima
        perfil_horizontal = np.sum(roi, axis=0)
        picos = len([1 for i in range(1, len(perfil_horizontal)-1)
                    if perfil_horizontal[i] > perfil_horizontal[i-1]
                    and perfil_horizontal[i] > perfil_horizontal[i+1]])

        return picos >= self.config['picos_minimos']

    return False

def _calcular_score_placa(self, roi, area, aspect_ratio):
    """Calcular score de qualidade para candidato"""
    score = 0.0

    # Score por aspect ratio (ideal ~3.5:1)
    aspect_ideal = 3.5
    aspect_diff = abs(aspect_ratio - aspect_ideal) / aspect_ideal
    score += max(0, 1.0 - aspect_diff) * 0.3

    # Score por área (ideal ~10000 pixels)
    area_ideal = 10000
    area_diff = abs(area - area_ideal) / area_ideal
    score += max(0, 1.0 - area_diff) * 0.2

    # Score por características de texto
    if roi.size > 0:
        densidade = np.sum(roi == 255) / roi.size
        if 0.3 <= densidade <= 0.7:  # Densidade ideal
            score += 0.3

        # Regularidade do padrão
        if len(roi.shape) == 2 and roi.shape[1] > 5:
            perfil = np.sum(roi, axis=0)
            if len(perfil) > 0 and np.mean(perfil) > 0:
                variancia_norm = np.std(perfil) / (np.mean(perfil) + 1)
                if 0.5 <= variancia_norm <= 2.0:
                    score += 0.2

    return min(score, 1.0)

def _filtrar_placas_candidatas(self, candidatos):
    """Eliminar duplicatas e ranquear por qualidade"""
    if not candidatos:
        return []

    # Remover duplicatas (IoU > 0.5)
    candidatos_unicos = []

    for candidato in candidatos:
        x1, y1, x2, y2 = candidato['bbox']

        duplicata = False
        for unico in candidatos_unicos:
            ux1, uy1, ux2, uy2 = unico['bbox']

            # Calcular IoU
            inter_x1, inter_y1 = max(x1, ux1), max(y1, uy1)
            inter_x2, inter_y2 = min(x2, ux2), min(y2, uy2)

            if inter_x1 < inter_x2 and inter_y1 < inter_y2:
                inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
                area1 = (x2 - x1) * (y2 - y1)
                area2 = (ux2 - ux1) * (uy2 - uy1)
                iou = inter_area / (area1 + area2 - inter_area)

                if iou > 0.5:  # Sobreposição significativa
                    duplicata = True
                    if candidato['score'] > unico['score']:
                        candidatos_unicos.remove(unico)
                        candidatos_unicos.append(candidato)
                    break

        if not duplicata:
            candidatos_unicos.append(candidato)

    # Ordenar por score
    candidatos_unicos.sort(key=lambda x: x['score'], reverse=True)
    return candidatos_unicos

def _validar_com_ocr_preliminar(self, candidatos, imagem_original):
    """Validação com OCR preliminar"""
    placas_validadas = []

    for candidato in candidatos:
        x1, y1, x2, y2 = candidato['bbox']
        roi = imagem_original[y1:y2, x1:x2]

        if roi.size == 0:
            continue

        # OCR rápido
        texto_easyocr = self._ocr_rapido_easyocr(roi)
        texto_tesseract = self._ocr_rapido_tesseract(roi)

        # Validar formato brasileiro
        score_texto = self._validar_texto_placa(texto_easyocr, texto_tesseract)

        if score_texto > 0.3:  # Threshold para validação
            candidato['confianca'] = candidato['score'] * 0.7 + score_texto * 0.3
            candidato['imagem_placa'] = roi
            candidato['texto_preliminar'] = texto_easyocr or texto_tesseract
            placas_validadas.append(candidato)

    return placas_validadas

def _ocr_rapido_easyocr(self, imagem):
    """OCR EasyOCR rápido"""
    if self.easyocr_reader is None:
        return ""
    try:
        results = self.easyocr_reader.readtext(imagem, verbose=False)
        return "".join([res[1] for res in results if res[2] > 0.3]).replace(' ', '').upper()
    except:
        return ""

def _ocr_rapido_tesseract(self, imagem):
    """OCR Tesseract rápido"""
    try:
        if len(imagem.shape) == 3:
            imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
        texto = pytesseract.image_to_string(imagem, config=self.config['tesseract_config'])
        return texto.strip().replace(' ', '').replace('\n', '').upper()
    except:
        return ""

def _validar_texto_placa(self, texto1, texto2):
    """Validar se texto tem características de placa brasileira"""
    textos = [t for t in [texto1, texto2] if t]
    if not textos:
        return 0.0

    melhor_score = 0.0

    for texto in textos:
        score = 0.0

        # Tamanho típico (5-8 caracteres)
        if 5 <= len(texto) <= 8:
            score += 0.3
        if len(texto) == 7:  # Tamanho exato brasileiro
            score += 0.2

        # Proporção letras vs números
        letras = sum(1 for c in texto if c.isalpha())
        numeros = sum(1 for c in texto if c.isdigit())

        if letras >= 2 and numeros >= 2:
            score += 0.3

        # Padrão brasileiro específico
        if len(texto) == 7:
            if texto[:3].isalpha() and texto[3:].isdigit():
                score += 0.4  # ABC1234
            elif (texto[:3].isalpha() and texto[3].isdigit() and
                  texto[4].isalpha() and texto[5:].isdigit()):
                score += 0.4  # ABC1D23 (Mercosul)

        # Diversidade de caracteres
        if len(set(texto)) >= len(texto) * 0.4:
            score += 0.1

        melhor_score = max(melhor_score, score)

    return min(melhor_score, 1.0)

# Adicionar métodos restantes à classe
SistemaReconhecimentoPlacasMelhorado._tem_caracteristicas_texto = _tem_caracteristicas_texto
SistemaReconhecimentoPlacasMelhorado._calcular_score_placa = _calcular_score_placa
SistemaReconhecimentoPlacasMelhorado._filtrar_placas_candidatas = _filtrar_placas_candidatas
SistemaReconhecimentoPlacasMelhorado._validar_com_ocr_preliminar = _validar_com_ocr_preliminar
SistemaReconhecimentoPlacasMelhorado._ocr_rapido_easyocr = _ocr_rapido_easyocr
SistemaReconhecimentoPlacasMelhorado._ocr_rapido_tesseract = _ocr_rapido_tesseract
SistemaReconhecimentoPlacasMelhorado._validar_texto_placa = _validar_texto_placa

## 5️⃣ Método de Processamento Principal

In [ ]:
def processar_imagem_melhorado(self, caminho_imagem):
    """Processar imagem com sistema melhorado"""
    imagem = cv2.imread(caminho_imagem)
    if imagem is None:
        return {'erro': f'Não foi possível carregar: {caminho_imagem}'}

    nome_arquivo = os.path.basename(caminho_imagem)

    resultado = {
        'caminho': caminho_imagem,
        'nome_arquivo': nome_arquivo,
        'imagem_original': imagem,
        'preprocessing': {},
        'placas_detectadas': [],
        'resultados_ocr': []
    }

    # ETAPA 1: Filtrar região de interesse
    imagem_roi, _ = self.filtrar_regiao_interesse(imagem)

    # ETAPA 2: Pré-processamento otimizado
    resultado['preprocessing'] = self.preprocessar_para_placas(imagem_roi)

    # ETAPA 3: Detecção melhorada (múltiplas estratégias)
    placas = self.detectar_placas_melhorado(imagem)
    resultado['placas_detectadas'] = placas

    print(f" {len(placas)} placa(s) detectada(s) com alta confiança")

    if placas:
        for placa in placas:
            metodo = placa.get('metodo', 'N/A')
            score = placa.get('score', 0)
            confianca = placa.get('confianca', 0)
            w = placa['bbox'][2] - placa['bbox'][0]
            h = placa['bbox'][3] - placa['bbox'][1]
            print(f"    {metodo} - Score: {score:.3f} - Conf: {confianca:.3f} - Dim: {w}x{h}")

    # ETAPA 4: OCR detalhado para cada placa
    for i, placa in enumerate(placas):
        print(f"\n OCR detalhado na placa {i+1}...")

        imagem_placa = placa['imagem_placa']

        # Pré-processar placa individual
        prep_placa = self.preprocessar_para_placas(imagem_placa)

        # OCR completo com múltiplas estratégias
        texto_tesseract = self._ocr_tesseract_completo(prep_placa['morph_opening'])
        texto_easyocr = self._ocr_easyocr_completo(imagem_placa)

        # Pós-processamento
        final_tesseract = self._pos_processar_texto(texto_tesseract)
        final_easyocr = self._pos_processar_texto(texto_easyocr)

        resultado_ocr = {
            'placa_id': i,
            'bbox': placa['bbox'],
            'confianca_deteccao': placa.get('confianca', placa.get('score', 0)),
            'metodo_deteccao': placa.get('metodo', 'N/A'),
            'score_qualidade': placa.get('score', 0),
            'dimensoes': f"{placa['bbox'][2] - placa['bbox'][0]}x{placa['bbox'][3] - placa['bbox'][1]}",
            'aspect_ratio': placa.get('aspect_ratio', 0),
            'area': placa.get('area', 0),
            'imagem_placa': imagem_placa,
            'preprocessing_placa': prep_placa,
            'tesseract': {
                'texto_bruto': texto_tesseract,
                'texto_final': final_tesseract
            },
            'easyocr': {
                'texto_bruto': texto_easyocr,
                'texto_final': final_easyocr
            }
        }

        resultado['resultados_ocr'].append(resultado_ocr)

        print(f"   Tesseract: '{final_tesseract}'")
        print(f"   EasyOCR: '{final_easyocr}'")
        print(f"   Score: {placa.get('score', 0):.3f} | Conf: {placa.get('confianca', 0):.3f}")
        print(f"   {placa['bbox'][2] - placa['bbox'][0]}x{placa['bbox'][3] - placa['bbox'][1]} (ratio: {placa.get('aspect_ratio', 0):.2f})")

    return resultado

def _ocr_tesseract_completo(self, imagem):
    """OCR Tesseract com múltiplas estratégias"""
    if len(imagem.shape) == 3:
        imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

    resultados = []

    # Estratégia 1: Original
    try:
        texto1 = pytesseract.image_to_string(imagem, config=self.config['tesseract_config'])
        resultados.append(texto1.strip().replace(' ', '').replace('\n', ''))
    except:
        pass

    # Estratégia 2: Redimensionada (2x)
    try:
        imagem_2x = cv2.resize(imagem, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        texto2 = pytesseract.image_to_string(imagem_2x, config=self.config['tesseract_config'])
        resultados.append(texto2.strip().replace(' ', '').replace('\n', ''))
    except:
        pass

    # Estratégia 3: Com denoising
    try:
        imagem_clean = cv2.fastNlMeansDenoising(imagem)
        texto3 = pytesseract.image_to_string(imagem_clean, config=self.config['tesseract_config'])
        resultados.append(texto3.strip().replace(' ', '').replace('\n', ''))
    except:
        pass

    return max(resultados, key=len) if resultados else ""

def _ocr_easyocr_completo(self, imagem):
    """OCR EasyOCR otimizado"""
    if self.easyocr_reader is None:
        return ""

    try:
        results = self.easyocr_reader.readtext(
            imagem,
            verbose=False,
            paragraph=False,
            width_ths=0.7,
            height_ths=0.7
        )

        texto_final = ""
        for (bbox, texto, confianca) in results:
            if confianca > 0.4:  # Threshold otimizado
                texto_final += texto

        return texto_final.replace(' ', '').upper()
    except:
        return ""

def _pos_processar_texto(self, texto):
    """Pós-processamento de texto"""
    chars_validos = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    texto_limpo = ''.join([c for c in texto.upper() if c in chars_validos])

    # Correções comuns de OCR
    # Correções comuns de OCR em reconhecimento de placas de veículos
    correções = {
        # Letras ↔ Números semelhantes
        '0': 'O', 'O': '0',
        '1': 'I', 'I': '1', 'L': '1', '|': '1',
        '2': 'Z', 'Z': '2',
        '3': 'E', 'E': '3',
        '4': 'A', 'A': '4',
        '5': 'S', 'S': '5',
        '6': 'G', 'G': '6',
        '7': 'T', 'T': '7',
        '8': 'B', 'B': '8',
        '9': 'P', 'P': '9', 'q': '9', 'Q': '9',
        '3': 'S', 'S': '3',

        # Casos extras (erros comuns de OCR)
        'D': '0', 'Q': '0',
        'C': 'G', 'G': 'C',
        'M': 'N', 'N': 'M',
        'U': 'V', 'V': 'U',
        'Y': 'V', 'V': 'Y',
        'R': 'K', 'K': 'R'
    }


    # Formatação brasileira
    if len(texto_limpo) == 7:
        return f"{texto_limpo[:3]}-{texto_limpo[3:]}"

    return texto_limpo

# Adicionar métodos à classe
SistemaReconhecimentoPlacasMelhorado.processar_imagem = processar_imagem_melhorado
SistemaReconhecimentoPlacasMelhorado._ocr_tesseract_completo = _ocr_tesseract_completo
SistemaReconhecimentoPlacasMelhorado._ocr_easyocr_completo = _ocr_easyocr_completo
SistemaReconhecimentoPlacasMelhorado._pos_processar_texto = _pos_processar_texto


## 6️⃣ Função de Visualização Melhorada

In [ ]:
def visualizar_resultados_sistema_melhorado(resultado, mostrar_preprocessing=True):
    """Visualização completa dos resultados melhorados"""
    if 'erro' in resultado:
        print(f"❌ {resultado['erro']}")
        return

    nome = resultado['nome_arquivo']
    imagem_original = resultado['imagem_original']

    print("=" * 70)

    if mostrar_preprocessing and resultado['preprocessing']:
        # Mostrar pipeline de pré-processamento
        plt.figure(figsize=(24, 12))

        prep = resultado['preprocessing']

        # Row 1: Pré-processamento
        plt.subplot(2, 5, 1)
        plt.imshow(cv2.cvtColor(imagem_original, cv2.COLOR_BGR2RGB))
        plt.title('1. Original\n Imagem de Entrada', fontsize=11, fontweight='bold')
        plt.axis('off')

        plt.subplot(2, 5, 2)
        plt.imshow(cv2.cvtColor(prep['suavizada'], cv2.COLOR_BGR2RGB))
        plt.title('2. Suavizada\n Filtro Bilateral', fontsize=11, fontweight='bold')
        plt.axis('off')

        plt.subplot(2, 5, 3)
        plt.imshow(prep['bin_adaptiva'], cmap='gray')
        plt.title('3. Binarizada\n Threshold Adaptivo', fontsize=11, fontweight='bold')
        plt.axis('off')

        plt.subplot(2, 5, 4)
        plt.imshow(prep['bordas_canny'], cmap='gray')
        plt.title('4. Bordas Canny\n Detecção Otimizada', fontsize=11, fontweight='bold')
        plt.axis('off')

        plt.subplot(2, 5, 5)
        plt.imshow(prep['bin_adaptiva_inv'], cmap='gray')
        plt.title('5. Binarizada Inversa\n Para Placas Escuras', fontsize=11, fontweight='bold')
        plt.axis('off')

        # Row 2: Morfologia e resultado
        plt.subplot(2, 5, 6)
        plt.imshow(prep['morph_close_horizontal'], cmap='gray')
        plt.title('6. Morfologia H\n↔ Conectar Caracteres', fontsize=11, fontweight='bold')
        plt.axis('off')

        plt.subplot(2, 5, 7)
        plt.imshow(prep['morph_opening'], cmap='gray')
        plt.title('7. Morfologia Final\n Limpeza de Ruído', fontsize=11, fontweight='bold')
        plt.axis('off')

        # Resultado final com detecções PRECISAS
        imagem_resultado = imagem_original.copy()

        for i, resultado_ocr in enumerate(resultado['resultados_ocr']):
            x1, y1, x2, y2 = resultado_ocr['bbox']

            # Bounding box VERDE GROSSO para placas
            cv2.rectangle(imagem_resultado, (x1, y1), (x2, y2), (0, 255, 0), 4)

            # Texto reconhecido
            texto_final = resultado_ocr['easyocr']['texto_final']
            if not texto_final:
                texto_final = resultado_ocr['tesseract']['texto_final']

            if texto_final:
                # Fundo verde para texto
                cv2.rectangle(imagem_resultado, (x1, y1-40), (x1+280, y1), (0, 255, 0), -1)
                cv2.putText(imagem_resultado, texto_final, (x1+5, y1-15),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)

            # Informações da detecção
            info = f"ID:{i+1} {resultado_ocr['metodo_deteccao']}"
            cv2.putText(imagem_resultado, info, (x1, y2+20),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            score = resultado_ocr['score_qualidade']
            cv2.putText(imagem_resultado, f"Score: {score:.2f}", (x1, y2+40),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        plt.subplot(2, 5, 8)
        plt.imshow(cv2.cvtColor(imagem_resultado, cv2.COLOR_BGR2RGB))
        plt.title('8. PLACAS DETECTADAS\n Sistema ', fontsize=11, fontweight='bold', color='green')
        plt.axis('off')

        plt.subplot(2, 5, 10)
        plt.axis('off')
        melhorias_text = ''
        plt.text(0.05, 0.95, melhorias_text, transform=plt.gca().transAxes,
                fontsize=9, verticalalignment='top', color='green', fontweight='bold')

        plt.suptitle(f'Detecção Precisa de Placas - {nome}',
                     fontsize=16, fontweight='bold', color='darkgreen')
        plt.tight_layout()
        plt.show()

    # Detalhes das placas detectadas
    if resultado['resultados_ocr']:
        print(f"\n PLACAS DETECTADAS ({len(resultado['resultados_ocr'])})")
        print("-" * 70)

        for i, resultado_ocr in enumerate(resultado['resultados_ocr']):
            print(f"    Confiança: {resultado_ocr['confianca_deteccao']:.3f}")
            print(f"    Score Qualidade: {resultado_ocr['score_qualidade']:.3f}")
            print(f"    Área: {resultado_ocr['area']} pixels")
            print(f"    Tesseract: '{resultado_ocr['tesseract']['texto_final']}'")
            print(f"    EasyOCR: '{resultado_ocr['easyocr']['texto_final']}'")

            # Mostrar processamento individual da placa
            plt.figure(figsize=(15, 5))

            plt.subplot(1, 4, 1)
            plt.imshow(cv2.cvtColor(resultado_ocr['imagem_placa'], cv2.COLOR_BGR2RGB))
            plt.title(f'Placa {i+1}\n Original\n{resultado_ocr["metodo_deteccao"]}',
                     fontweight='bold', fontsize=10)
            plt.axis('off')

            plt.subplot(1, 4, 2)
            plt.imshow(resultado_ocr['preprocessing_placa']['bin_adaptiva'], cmap='gray')
            plt.title(f'Placa {i+1}\n Binarizada\nScore: {resultado_ocr["score_qualidade"]:.3f}',
                     fontweight='bold', fontsize=10)
            plt.axis('off')

            plt.subplot(1, 4, 3)
            plt.imshow(resultado_ocr['preprocessing_placa']['morph_close_horizontal'], cmap='gray')
            plt.title(f'Placa {i+1}\n↔ Morfologia H\nÁrea: {resultado_ocr["area"]}px',
                     fontweight='bold', fontsize=10)
            plt.axis('off')

            plt.subplot(1, 4, 4)
            plt.imshow(resultado_ocr['preprocessing_placa']['morph_opening'], cmap='gray')
            plt.title(f'Placa {i+1}\n Para OCR\nConf: {resultado_ocr["confianca_deteccao"]:.3f}',
                     fontweight='bold', fontsize=10)
            plt.axis('off')

            plt.suptitle(f'Processamento Detalhado da Placa {i+1}',
                        fontsize=14, fontweight='bold', color='darkgreen')
            plt.tight_layout()
            plt.show()
    else:
        print("\n  Nenhuma placa detectada.")

## 7️⃣ Função Principal de Execução

In [ ]:
def executar_sistema_melhorado():
    """Executar sistema melhorado de reconhecimento de placas"""

    # Inicializar sistema melhorado
    sistema = SistemaReconhecimentoPlacasMelhorado()

    # Verificar imagens
    pasta_imagens = '/content/drive/MyDrive/placas_veiculos'
    extensoes = ['.jpg', '.jpeg', '.png', '.bmp']

    imagens_encontradas = []
    if os.path.exists(pasta_imagens):
        for ext in extensoes:
            imagens_encontradas.extend([
                os.path.join(pasta_imagens, f)
                for f in os.listdir(pasta_imagens)
                if f.lower().endswith(ext.lower())
            ])

    if not imagens_encontradas:
        print("\n  NENHUMA IMAGEM ENCONTRADA!")
        print(f" Pasta: {pasta_imagens}")
        return

    print(f"\n {len(imagens_encontradas)} imagem(s) encontrada(s) para processamento")
    print(" Iniciando processamento com sistema melhorado...")
    print("-" * 70)

    # Processar cada imagem
    resultados_completos = []
    placas_total = 0
    deteccoes_precisas = 0

    for i, caminho_img in enumerate(imagens_encontradas):
        print(f"\n[{i+1}/{len(imagens_encontradas)}] " + "=" * 50)

        try:
            # Processar com sistema melhorado
            resultado = sistema.processar_imagem(caminho_img)
            resultados_completos.append(resultado)

            if 'erro' not in resultado:
                num_placas = len(resultado['resultados_ocr'])
                placas_total += num_placas

                # Contar detecções precisas (score > 0.5)
                precisas = sum(1 for r in resultado['resultados_ocr']
                              if r['score_qualidade'] > 0.5)
                deteccoes_precisas += precisas

                print(f"{num_placas} placa(s) detectada(s) - {precisas}")

                # Visualizar resultado melhorado
                visualizar_resultados_sistema_melhorado(resultado, mostrar_preprocessing=True)

        except Exception as e:
            print(f"ERRO: {e}")
            import traceback
            traceback.print_exc()

    # RELATÓRIO FINAL MELHORADO

    imagens_processadas = len([r for r in resultados_completos if 'erro' not in r])
    imagens_erro = len([r for r in resultados_completos if 'erro' in r])
    taxa_precisao = (deteccoes_precisas / max(placas_total, 1)) * 100

    print(f"\n ESTATÍSTICAS:")
    print(f"   • Imagens processadas: {imagens_processadas}")
    print(f"   • Imagens com erro: {imagens_erro}")
    print(f"   • Total de placas detectadas: {placas_total}")
    print(f"   • Detecções de alta precisão: {deteccoes_precisas}")
    print(f"   • Taxa de precisão: {taxa_precisao:.1f}%")

    if placas_total > 0:
        print(f"\n RESUMO DE RECONHECIMENTO:")

        for i, resultado in enumerate(resultados_completos):
            if 'erro' not in resultado and resultado['resultados_ocr']:
                nome = resultado['nome_arquivo']
                print(f"\n  {nome}:")

                for j, ocr_result in enumerate(resultado['resultados_ocr']):
                    metodo = ocr_result['metodo_deteccao']
                    score = ocr_result['score_qualidade']
                    conf = ocr_result['confianca_deteccao']
                    dimensoes = ocr_result['dimensoes']
                    tesseract = ocr_result['tesseract']['texto_final']
                    easyocr = ocr_result['easyocr']['texto_final']

                    status = "-" if score > 0.5 else "⚠️"

                    print(f"   {status} Placa {j+1} ({metodo}):")
                    print(f"       {dimensoes} | Score: {score:.3f} | Conf: {conf:.3f}")
                    print(f"       Tesseract: '{tesseract}'")
                    print(f"       EasyOCR: '{easyocr}'")


## ▶️ EXECUÇÃO DO CÓDIGO PRINCIPAL

In [ ]:
resultados_sistema_melhorado = executar_sistema_melhorado()